# Russell 2000 Rebalance Events — Data Engineering

Produces `russell_events.csv`: one row per classified add/delete from the Russell 2000 annual reconstitution.

**Classification logic:**
- **ADD / FROM_R1000**: entered RTY, was previously in RAY but not RTY (Russell 1000)
- **ADD / FROM_RMICRO**: entered RTY, was previously in Russell Microcap
- **DELETE / TO_R1000**: exited RTY, now in RAY but not RTY (promoted to Russell 1000)
- **DELETE / TO_RMICRO**: exited RTY, now in Russell Microcap

Stocks outside the known Russell universe (new IPOs, delisted, acquired) are excluded.

In [ ]:
import re
import pandas as pd
from pathlib import Path
from datetime import datetime

DATA_ROOT     = Path("Rebalance data")
CALENDAR_FILE = Path("Rebalance data/05072026_IRBv1_NB.xlsx")

In [ ]:
# --- Load Russell Calendar ---
_cal_raw = pd.read_excel(CALENDAR_FILE, sheet_name="Russell Calendar", header=3)
_cal_raw.columns = ["Year", "Period", "Rank_Date", "Announcement_Date", "Effective_Date", "Effective_Open", "Notes"]
russell_calendar = (
    _cal_raw[["Year", "Rank_Date", "Announcement_Date", "Effective_Date", "Effective_Open"]]
    .dropna(subset=["Year"])
    .assign(Year=lambda df: df["Year"].astype(int))
    .set_index("Year")
)
russell_calendar

In [ ]:
def load_index(path: Path) -> pd.DataFrame:
    """Load an index membership file, returning Ticker only."""
    df = pd.read_excel(path, sheet_name="Worksheet")
    return df[["Ticker"]].drop_duplicates()


def parse_file_date(path: Path) -> datetime:
    """Extract date from filenames like 'RTY as of Jun 27 20251_xxx.xlsx'."""
    m = re.search(r"as of (\w+) (\d+) (\d{4})", path.stem, re.IGNORECASE)
    if m:
        return datetime.strptime(f"{m.group(1)} {m.group(2)} {m.group(3)}", "%b %d %Y")
    raise ValueError(f"Cannot parse date from filename: {path.name}")


def get_pre_post(year_path: Path, prefix: str):
    """
    Return (pre_file, post_file) for a given index prefix in a year folder.
    Deduplicates files that share the same date (keeps one per date).
    If only one unique date exists, returns that file for both pre and post.
    """
    files = [
        f for f in year_path.glob(f"{prefix} as of *.xlsx")
        if not f.name.startswith("~$")
    ]
    # Deduplicate: one file per unique date (take the shortest filename as canonical)
    by_date = {}
    for f in files:
        d = parse_file_date(f)
        if d not in by_date or len(f.name) < len(by_date[d].name):
            by_date[d] = f
    unique = sorted(by_date.values(), key=parse_file_date)

    if len(unique) == 2:
        return unique[0], unique[1]
    elif len(unique) == 1:
        return unique[0], unique[0]
    else:
        raise ValueError(f"Expected 1–2 unique dates for {prefix} in {year_path}, found {len(unique)}")

In [ ]:
def process_year(year_path: Path) -> pd.DataFrame:
    """
    Classify all Russell 2000 adds and deletes for a single rebalance year.
    Dates are sourced from the Russell Calendar sheet.

    Returns
    -------
    DataFrame with columns: Ticker, Event_Type, Classification,
                             Rank_Date, Announcement_Date, Effective_Date, Effective_Open, Year
    """
    year = int(year_path.name)
    if year not in russell_calendar.index:
        raise ValueError(f"Year {year} not found in Russell Calendar")
    cal = russell_calendar.loc[year]

    # --- Load all six membership files ---
    rty_pre_path,  rty_post_path  = get_pre_post(year_path, "RTY")
    ray_pre_path,  ray_post_path  = get_pre_post(year_path, "RAY")
    rmi_pre_path,  rmi_post_path  = get_pre_post(year_path, "RMICRO")

    rty_pre  = load_index(rty_pre_path)
    rty_post = load_index(rty_post_path)
    ray_pre  = load_index(ray_pre_path)
    ray_post = load_index(ray_post_path)
    rmi_pre  = load_index(rmi_pre_path)
    rmi_post = load_index(rmi_post_path)

    print(f"[{year}] RTY pre={len(rty_pre):,}  post={len(rty_post):,}")
    print(f"[{year}] RAY pre={len(ray_pre):,}  post={len(ray_post):,}")
    print(f"[{year}] RMICRO pre={len(rmi_pre):,}  post={len(rmi_post):,}")

    # --- Build ticker sets ---
    rty_pre_set  = set(rty_pre["Ticker"]);  rty_post_set = set(rty_post["Ticker"])
    ray_pre_set  = set(ray_pre["Ticker"]);  ray_post_set = set(ray_post["Ticker"])
    rmi_pre_set  = set(rmi_pre["Ticker"]);  rmi_post_set = set(rmi_post["Ticker"])

    # Russell 1000 = RAY minus RTY (derived)
    r1000_pre_set  = ray_pre_set  - rty_pre_set
    r1000_post_set = ray_post_set - rty_post_set

    adds    = rty_post_set - rty_pre_set
    deletes = rty_pre_set  - rty_post_set

    # --- Classify adds ---
    rows = []
    for tkr in adds:
        if tkr in r1000_pre_set:     clf = "FROM_R1000"
        elif tkr in rmi_pre_set:     clf = "FROM_RMICRO"
        else:                        continue
        rows.append({"Ticker": tkr, "Event_Type": "ADD", "Classification": clf})

    # --- Classify deletes ---
    for tkr in deletes:
        if tkr in r1000_post_set:    clf = "TO_R1000"
        elif tkr in rmi_post_set:    clf = "TO_RMICRO"
        else:                        continue
        rows.append({"Ticker": tkr, "Event_Type": "DELETE", "Classification": clf})

    print(f"[{year}] Adds: {len(adds)} total → {sum(1 for r in rows if r['Event_Type']=='ADD')} classified")
    print(f"[{year}] Deletes: {len(deletes)} total → {sum(1 for r in rows if r['Event_Type']=='DELETE')} classified")

    # --- Assemble DataFrame ---
    df = pd.DataFrame(rows)
    df["Rank_Date"]         = pd.to_datetime(cal["Rank_Date"])
    df["Announcement_Date"] = pd.to_datetime(cal["Announcement_Date"])
    df["Effective_Date"]    = pd.to_datetime(cal["Effective_Date"])
    df["Effective_Open"]    = pd.to_datetime(cal["Effective_Open"])
    df["Year"] = year

    return df[["Ticker", "Event_Type", "Classification",
               "Rank_Date", "Announcement_Date", "Effective_Date", "Effective_Open", "Year"]]

In [ ]:
# --- Auto-discover all year folders under DATA_ROOT and process each ---
year_folders = sorted(
    [p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.isdigit()],
    key=lambda p: int(p.name)
)
print("Year folders found:", [p.name for p in year_folders])

all_events = pd.concat(
    [process_year(p) for p in year_folders],
    ignore_index=True
)

In [ ]:
# --- Sanity checks ---
print("Total rows:", len(all_events))
print()
print("Event_Type breakdown:")
print(all_events["Event_Type"].value_counts())
print()
print("Classification breakdown:")
print(all_events["Classification"].value_counts())
print()
print("Null check:")
print(all_events[["Ticker", "Event_Type", "Classification", "Effective_Date"]].isnull().sum())

## Russell Universe CSV

Separate output: every ticker seen across **all** index files for all years, with a flag indicating whether it was in the R3000E (RAY ∪ RMICRO) in the **pre-rebalance** snapshot for that year.

In [ ]:
def build_universe_year(year_path: Path) -> pd.DataFrame:
    """
    For a single year, collect every ticker seen across all 6 index files
    and flag whether it was in the R3000E pre-rebalance snapshot.

    R3000E pre = RAY_pre ∪ RMICRO_pre
    """
    year = int(year_path.name)

    rty_pre_path,  rty_post_path  = get_pre_post(year_path, "RTY")
    ray_pre_path,  ray_post_path  = get_pre_post(year_path, "RAY")
    rmi_pre_path,  rmi_post_path  = get_pre_post(year_path, "RMICRO")

    ray_pre  = load_index(ray_pre_path)
    ray_post = load_index(ray_post_path)
    rty_pre  = load_index(rty_pre_path)
    rty_post = load_index(rty_post_path)
    rmi_pre  = load_index(rmi_pre_path)
    rmi_post = load_index(rmi_post_path)

    all_tickers = (
        pd.concat([ray_pre, ray_post, rty_pre, rty_post, rmi_pre, rmi_post])
        ["Ticker"].drop_duplicates().reset_index(drop=True)
    )

    r3000e_pre = set(ray_pre["Ticker"]) | set(rmi_pre["Ticker"])

    df = pd.DataFrame({"Ticker": all_tickers})
    df["Year"] = year
    df["In_R3000E_Pre_Rebalance"] = df["Ticker"].isin(r3000e_pre)

    return df[["Ticker", "Year", "In_R3000E_Pre_Rebalance"]]


universe = pd.concat(
    [build_universe_year(p) for p in year_folders],
    ignore_index=True
)

print(f"Total rows: {len(universe):,}")
print(f"Unique tickers: {universe['Ticker'].nunique():,}")
print()
print("In_R3000E_Pre_Rebalance breakdown:")
print(universe["In_R3000E_Pre_Rebalance"].value_counts())
universe.head()

In [ ]:
universe_path = Path("../russell_universe.csv")
universe.to_csv(universe_path, index=False)
print(f"Saved {len(universe):,} rows to {universe_path.resolve()}")

In [ ]:
# All unique tickers across all years — one column
all_tickers = universe[["Ticker"]].drop_duplicates().sort_values("Ticker").reset_index(drop=True)
all_tickers.to_excel("../russell_tickers.xlsx", index=False)
print(f"Saved {len(all_tickers):,} unique tickers to russell_tickers.xlsx")

In [ ]:
# --- Save ---
out_path = Path("russell_events.csv")
all_events.to_csv(out_path, index=False)
print(f"Saved {len(all_events)} rows to {out_path.resolve()}")
all_events